#1. data loading and imports loading for further use

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from scipy.stats import chi2_contingency
from scipy.stats import f_oneway
from scipy.stats import pearsonr


In [ ]:
df = pd.read_csv("insurance_claims.csv")

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.info

In [ ]:
df.shape

In [ ]:
df.describe()

#2 data cleaning

In [ ]:
df.isnull().sum()

In [ ]:
df = df.drop(columns=["_c39"])

In [ ]:
df.shape

In [ ]:
def show_formatted(df, decimals=2):
    float_cols = df.select_dtypes(include="float").columns
    format_dict = {col: f"{{:.{decimals}f}}" for col in float_cols}
    display(df.style.format(format_dict).hide(axis="index"))

In [ ]:
df["fraud_reported"] = df["fraud_reported"].map({
    "Y": "Fraud",
    "N": "No Fraud"
})
df["property_damage"] = df["property_damage"].map({
    "?": "Unknown Damage",
    "YES": "Property Damage",
    "NO": "No Property Damage"
})
df["collision_type"] = df["collision_type"].replace("?", "Unknown Collision")

#3 SQL analysis

In [ ]:
import sqlite3

In [ ]:
conn = sqlite3.connect("insurance.db")

In [ ]:
df.to_sql("claims", conn, if_exists="replace", index=False)

In [ ]:
import os
print(os.getcwd())

In [ ]:
import os
print(os.listdir())

In [ ]:
pd.read_sql_query("SELECT COUNT(*) AS rows FROM claims;", conn)

Formatting and Explanation of data choices: 

Bins: I chose to use the square root method to calculate bins as it avoids the failures of many other bin types and is simple to implement. Stuge's rule relies on claims being normally distributed and symmetric, which typically insurance claims are not, additionally it would use too little bins to accurately show the data. Freedman-Diaconis Rule, the initial binning method I set to use and the method which is typically good for more skewed data sets, would have too large of bins due to the large IQR of many of the variables. One note with the method chosen is that it will not adapt to large spikes or very long tails. 

Rounding in Charts: I chose to round decimals to 2 places to make charts easier to read

Variables: 
1. temp_df: this variable holds the data from each sql query and then uses it to make tables/ graphs. I chose to have only one temporary variable because I found that due to the 39 variables and the many analyses that needed to be run making a new variable each time would be too tedious and confusing. 
2. avg_claim_amount: used to average of total_claim_amount over other variables
3. claim_count: used to show the number of claims for each variable
4. total_losses: used to show the total losses from a given variable

Part 1: Understanding loss
Research Question: What factors are associated with loss?

1a. General Loss Trends 

In [ ]:
query = """
SELECT total_claim_amount
FROM claims;
"""
temp_df = pd.read_sql_query(query, conn)

#histogram
plt.hist(temp_df["total_claim_amount"], bins="sqrt", edgecolor="black")
plt.title("Distribution of Total Claim Amounts")
plt.xlabel("Injury Claim Amount ($)")
plt.ylabel("Number of Claims")
plt.show()


In [ ]:
cols = ["total_claim_amount"]
summary = df[cols].agg(["mean", "median", "std", "min", "max"]).T
q1 = df[cols].quantile(0.25)
q3 = df[cols].quantile(0.75)
iqr = q3 - q1
print(summary)

Claim amounts in general appear to follow a mixed distribution with about 10% of the claims near zero and 90% of the claims being roughly normally distributed with a mean around $60,0000. This means that there there is a sizeable amount of claims occuring around zero, likely due to some similar characteric of these claims, but that mostly when claims occur they have a relatively large claim amount. The median is slightly higher than the mean, due to the 10% of data having much lower claims on average than the rest of the data. 

1b. Incident Severity

In [ ]:
query = """
SELECT incident_severity, 
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY incident_severity
ORDER BY total_losses DESC;
"""
temp_df = pd.read_sql_query(query, conn)

#bar chart
show_formatted(temp_df)
plt.bar(temp_df["incident_severity"], temp_df["average_claim_amount"])
plt.xlabel("Incident Severity")
plt.ylabel("Average Claim Amount")
plt.title("Average Claim Amount by Incident Severity")
plt.show()

I analyzed incident severity to see first of all if there were bounds were that distinguished each severity level from one another (which there were not) and to see what the averages were for each severity. I found that Major Damage had the highest average claim severity, followed very closely by Total loss. I also saw that Major Damage then Total Loss and then Minor Damage had the highest total loss amounts. However these loss amounts were all relatively close to one another especially when compared with Trivial Damage. Which had an average claim amount of around 9x less that the other severity levels. Trivial damage had the lowest claim counts; 9% of all claims, while Minor Damage had the highest; 35% of all claims, and Major and Total Damage had very similar claim counts.

1c. Number of Vehicles Involved

In [ ]:
query = """
SELECT number_of_vehicles_involved, 
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY number_of_vehicles_involved
ORDER BY number_of_vehicles_involved;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

#bar chart
plt.bar(temp_df["number_of_vehicles_involved"].astype(str), temp_df["average_claim_amount"])
plt.xlabel("Number of Vehicles Involved")
plt.ylabel("Average Claim Amount ($)")
plt.title("Average Claim Amount by Number of Vehicles Involved")
plt.show()

In [ ]:
query = """
SELECT number_of_vehicles_involved, 
total_claim_amount
FROM claims;
"""
temp_df = pd.read_sql_query(query, conn)

#correlation 
corr = temp_df["number_of_vehicles_involved"].corr(temp_df["total_claim_amount"])
print(f"Correlation coefficient: {corr:.3f}")

This table shows that one and three car accidents make up the vast majority of accidents; specifically 94%, with one car accidents making up the most at 58%. However, one car accidents had the lowest average claim amount, around 75% the size of the other car accident types. The other car accident types also had similar average claim amounts. So the reason behind one car accidents having the largest total loss is due to the sheer amount of claims and not the claim amount. Additionally, the correlation is around 0.27 with a p value of 0, which implies a statistically significant weak positive relationship between number of vehicles involved and total claim amount.

1d. Bodily Injuries

In [ ]:
query = """
SELECT bodily_injuries, 
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY bodily_injuries
ORDER BY bodily_injuries;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

Bodily injuries were realitively equally as likely to occur, with similar claim counts as well as similar average claim amounts, and similar total losses.  This shows there is likely not a meaningful relationship between loss metrics and bodily injuries.

1e. Property Damage

In [ ]:
query = """
SELECT property_damage, 
AVG(total_claim_amount) AS average_severity, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY property_damage
ORDER BY average_severity DESC;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

For property damage claim counts, average severity, and total losses were relatively similar. This means that it likley does not share a meaningful relationship with loss metrics.

1f. Collision Type

In [ ]:
query = """
SELECT collision_type, 
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY collision_type
ORDER BY average_claim_amount DESC;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

#bar chart
plt.bar(temp_df["collision_type"], temp_df["average_claim_amount"])
plt.xlabel("Number of Vehicles Involved")
plt.ylabel("Average Claim Amount ($)")
plt.title("Average Claim Amount by Collision Type")
plt.show()

In [ ]:
query = """
SELECT collision_type, 
incident_severity,
COUNT(*) AS claim_count
FROM claims
GROUP BY incident_severity, collision_type;
"""
temp_df = pd.read_sql_query(query, conn)

#stacked bar chart
pivot = temp_df.pivot(index="collision_type", columns="incident_severity", values="claim_count")
pivot.plot(kind="bar", stacked=True)
plt.xlabel("Collision Type")
plt.ylabel("Claim Count")
plt.title("Claim Count by Incident Severity and Collision Type")
plt.legend(title="Incident Severity")
plt.show()


Front Collisions, Side Collisions, and Rear Collisions all had similar claim percentages and average severities, leading to a relatively uniform distribution of total losses across these three categories. However, the fourth category, unknown collision types, made up a much smaller share of claims and had a much lower average severity, only about 1/12 that of the other categories. This could imply that models using collision type should use better intake to classify unknown collisions into one of the other categories. This difference is because all of the unknown collisions consist of minor and trivial damages, which are the damages with the lowest average claim amounts. All of the other collision types are about evenly split between total loss, major loss, and minor damage, however none have any trivial damage. This is reflective of the pattern seen in loss severity where trivial damage had much lower average losses, followed by minor damage to a much lower degree.

1g. Incident Type

In [ ]:
query = """
SELECT incident_type, 
AVG(total_claim_amount) AS average_claim_amount,
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY incident_type
ORDER BY total_losses DESC;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

#bar chart
plt.bar(temp_df["incident_type"], temp_df["average_claim_amount"])
plt.xlabel("Incident Type")
plt.ylabel("Average Claim Amount")
plt.title("Average Claim Amount by Incident Type")
plt.xticks(rotation=45, ha="right")
plt.show()


In [ ]:
query = """
SELECT incident_type
FROM claims;
"""
temp_df = pd.read_sql_query(query, conn)

#pie chart
counts = temp_df["incident_type"].value_counts()
plt.pie(counts, labels=counts.index, autopct="%1.1f%%", startangle=90)
plt.title("Percentage of claims")
plt.show()


In [ ]:
query = """
SELECT incident_type, total_claim_amount
FROM claims
WHERE incident_type IN ('Multi-vehicle Collision', 'Single Vehicle Collision');
"""
temp_df = pd.read_sql_query(query, conn)

#t-test
multi = temp_df[temp_df["incident_type"] == "Multi-vehicle Collision"]["total_claim_amount"]
single = temp_df[temp_df["incident_type"] == "Single Vehicle Collision"]["total_claim_amount"]
t_stat, p_value = ttest_ind(multi, single)
print(f"T-statistic: {t_stat:.2f}")
print(f"p-value: {p_value:.4f}")

Average severity, claim count, and total losses varied substantially across incident types Single vehicle collisions and multi-vehicle collisions occurred more than 4x as often and had roughly 12x higher average severity than vehicle theft or parked car incidents, together driving total losses that were about 50 times larger for those two categories. Additionally, single vehicle and multi-vehicle collisions had very similar loss indicators, as did vehicle theft and parked car, suggesting incident type is closely associated with these loss metrics. However, despite appearing to have similar average claims after running a t-test to compare average claim amount of single vehicle vs multi-vehicle collisions I obtained a T-statistic of -2.52 and a p value of 0.0118. Implying that there is a significant difference in the average claim amount of these collision types.

Findings of Part 1: 

Overall, claim severity varied across some of the factors examined, while other variables showed relatively little difference in average claim amounts and loss indicators. Variables such as incident severity, number of cars involved, and collison type showed noticeable differences in claim severity, while property damage and bodily injuries showed relatively similar results across their levels. The distribution of incident severity and incident types was also generally consistent across these three variables. These results suggest that some characteristics of the claim or policy may be more closely associated with claim severity than others. Additionally the mixed distribution of claim distribution with 10% of the data falling very close to zero should be analyzed further to see possible explanations of this and why it differs from 90% of claim data which is approximately normally ditributed.

Part 2: Claim Components

Research Question: which type of claim is driving claim costs and do they differ fundamentally?

2a. Shape

In [ ]:
query = """
SELECT injury_claim, 
property_claim,
vehicle_claim
FROM claims;
"""
temp_df = pd.read_sql_query(query, conn)

#histogram injury claims
plt.hist(temp_df["injury_claim"], bins="sqrt", edgecolor="black")
plt.title("Distribution of Injury Claim Amounts")
plt.xlabel("Injury Claim Amount ($)")
plt.ylabel("Number of Claims")
plt.show()

#histogram property claims
plt.hist(temp_df["property_claim"], bins="sqrt", edgecolor="black")
plt.title("Distribution of Property Claim Amounts")
plt.xlabel("Injury Claim Amount ($)")
plt.ylabel("Number of Claims")
plt.show()

#histogram vehicle claims
plt.hist(temp_df["vehicle_claim"], bins="sqrt", edgecolor="black")
plt.title("Distribution of vehicle Claim Amounts")
plt.xlabel("Vehicle Claim Amount ($)")
plt.ylabel("Number of Claims")
plt.show()


All 3 claims follow a mixed distribution, with most of the claims centering around very low values and a smaller part of the data that is distributed differently. For both property and injury claims their second part of data is right skewed meaning that most non zero claims are moderate (a little bit more than $5000). Meanwhile vehicle claims have a second "hump" that is relatively normally distributed around $40,0000 to $50,0000, implying that most non zero claims fixate around this value. 

2b. Descriptive Statistics

In [ ]:
cols = ["injury_claim", "property_claim", "vehicle_claim"]
summary = temp_df[cols].agg(["mean", "median", "std", "min", "max"]).T
q1 = temp_df[cols].quantile(0.25)
q3 = temp_df[cols].quantile(0.75)
iqr = q3 - q1
summary["q1"] = q1
summary["q3"] = q3
summary["iqr"] = iqr
print(summary)

Injury claims and property claims both have very similar means that are 5.1x smaller than the mean of vehicle claims. Additionally, their medians are also extremely similar and 6.21x smaller than the median of vehicle claim which can be explained by skewed vs normal second distribution seen on both graphs. Additionally the minimum of both injury claim and property claim are zero, while the minimum of vehicle claim is 70, meaning that all auto insurance claims had some form of vehicle claim, while not all of them had injury or property claims. The standard deviation is also much larger for vehicle claims around 3.9x larger when compared to both injury and property claims. The interquartile range for vehicle claims is even larger at an aboutv 5x increase. So, vehicle claims are substantially more spread out especially in the middle which can be observed in the graphs as well.

2c. Composition of Total Claim Amounts

In [ ]:
query = """
SELECT injury_claim,
property_claim,
vehicle_claim
FROM claims;
"""
temp_df = pd.read_sql_query(query, conn)

#pie chart
avgs = temp_df[["injury_claim", "property_claim", "vehicle_claim"]].mean()
plt.pie(avgs, labels=["Injury", "Property", "Vehicle"], autopct="%1.1f%%", startangle=90)
plt.title("Average Composition of Total Claim Amount")
plt.show()

In [ ]:
query = """
SELECT injury_claim,
property_claim
FROM claims;
"""
temp_df = pd.read_sql_query(query, conn)

#no property claim %
temp_df[temp_df["property_claim"] == 0]
(temp_df["property_claim"] == 0).sum()
prop_0_property = (temp_df["property_claim"] == 0).mean()
temp_df[temp_df["injury_claim"] == 0]
(temp_df["injury_claim"] == 0).sum()
prop_0_injury = (temp_df["injury_claim"] == 0).mean()
print(f"The percentage of insurance claims with no property claim is:{prop_0_property:.1%}")
print(f"The percentage of insurance claims with no injury claim is:{prop_0_injury:.1%}")

In [ ]:
query = """
SELECT injury_claim, 
property_claim,
total_claim_amount
FROM claims;
"""
temp_df = pd.read_sql_query(query, conn)

#top ten largest claims %
ten_largest_injury = temp_df["injury_claim"].nlargest(10).sum() / temp_df["total_claim_amount"].sum()
ten_largest_property= temp_df["property_claim"].nlargest(10).sum() / temp_df["total_claim_amount"].sum()
print(f"The percentage of claim costs from the top 10 largest injury claims is:{ten_largest_injury:.1%}")
print(f"The percentage of claim costs from the top 10 largest property claims is :{ten_largest_property:.1%}")

Overall, the highest share of average (mean) claim cost comes from vehicle claims. These claims make up around 71.9% of the average claim cost, which is around 5x than either property or injury claims. Property and injury claims make up roughly the same percentage of claim costs. A very small percentage (1.9% and 2.5%) of property and injury claims respectively have no claims, meaning that most insurance claims have some amount of property and injury claims, but the actual dollar amount of these claims is very low. Similarly, only 0.4% of total claim cost comes from the top 10 claims for injury and property claims meaning that outliers do not disproprtionately affect claim costs for these variables.

Part 2 Summary: 

Overall, most insurance claim costs (70%) were driven by vehicle claims. Vehicle claims had much higher measures of spread and center. While all 3 claim types followed a mixed distribution, with a large "hump" of claims with low claim costs, vehicle claims had a secondary smaller "hump" centered around $40,000 to $50,000, while property and injury claims had almost identical distributions with a rightward skew and the majority of their second "hump" sitting just above $5000. This may imply that auto insurance for similar demographics to the one being analyzed should focus on reducing risk factors for vehicle damage, possibly by offering kick backs for cars with more safety features.

Part 3: Understanding Fraud

Research question: Is fraud more common amongst certain types of accidents?

3a. Loss Metrics and Fraud

In [ ]:
query = """
SELECT fraud_reported,
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*)*100.0/(SELECT COUNT(*) FROM claims) AS claim_percentage,
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_loss
FROM claims
GROUP BY fraud_reported
ORDER BY average_claim_amount DESC;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

#bar chart
plt.bar(temp_df["fraud_reported"], temp_df["average_claim_amount"])
plt.xlabel("Fraud Reported")
plt.ylabel("Average Claim Amount")
plt.title("Average Claim Amount by Fraud Reported")
plt.show()

In [ ]:
query = """
SELECT fraud_reported, total_claim_amount
FROM claims;
"""
temp_df = pd.read_sql_query(query, conn)

#t-test
fraudulent = temp_df[temp_df["fraud_reported"] == "Fraud"]["total_claim_amount"]
nonfradulent = temp_df[temp_df["fraud_reported"] == "No Fraud"]["total_claim_amount"]
t_stat, p_value = ttest_ind(fraudulent, nonfradulent)
print(f"T-statistic: {t_stat:.2f}")
print(f"p-value: {p_value:.4f}")

 Just under a quarter of claims were frauduelent and that these claims had an average claim amount that was $10000 higher than non fraudulent claims; roughly 1.19x. Running a t-test yeilded a T-statistic of 5.24 and a p-value of 0.0000, which suggests that this relationship is statistically significant. Overall, this suggests that although fraduelent claims make up a smaller proportion of claims, they have a higher average, suggesting they may represent a disproportionately large financial exposure. However, total loss was around 2.5x higher for incidents with reported fraud, meaning that most of this increase in total loss was driven by the sheer amount of claims, rather than an actual increase in average claim amount of the accidents. This could also be relevant for insurers as it could help identify claims that need more investigation, although claim cost alone is not sufficient in predicting fraud status.

3b. Types of Incidents Associated With Fraud

In [ ]:
query = """
SELECT fraud_reported,
incident_severity, 
COUNT(*) AS claim_count,
COUNT(*)*100.0/ SUM(COUNT(*)) OVER (PARTITION BY incident_severity) AS fraud_percentage
FROM claims
GROUP BY fraud_reported, incident_severity
ORDER by fraud_reported, incident_severity;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)


For claims where fraud was reported; the majority, 67.6% or about 2/3; are major damage, with trivial damage making up the least amount of claims at only 2.4%. However, for claims where no fraud was reported; major damage only made up 14% of claims, and minor damage and total loss make up the vast majority, at 74.3%. This suggests that major damage claims are disproportionately represented among claims where fraud was reported.

In [ ]:
query = """
SELECT fraud_reported,
incident_type,
COUNT(*) AS claim_count,
AVG(total_claim_amount) AS average_claim_amount,
COUNT(*)*100.0/SUM(COUNT(*)) OVER (PARTITION BY incident_type) AS fraud_percentage
FROM claims
GROUP BY incident_type, fraud_reported
ORDER BY incident_type;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

#stacked bar chart
pivot = temp_df.pivot(index="fraud_reported", columns="incident_type", values="claim_count")
pivot.plot(kind="bar", stacked=True)
plt.xlabel("Fraud Reported")
plt.ylabel("Claim Count")
plt.title("Claim Count by Incident Type and Fraud Reported")
plt.legend(title="Incident Type")
plt.show()



In [ ]:
query = """
SELECT incident_type,
incident_severity, 
COUNT(*) AS claim_count
FROM claims
GROUP BY incident_type, incident_severity
ORDER BY incident_severity, incident_type;
"""
temp_df = pd.read_sql_query(query, conn)

#stacked bar chart
pivot = temp_df.pivot(index="incident_severity", columns="incident_type", values="claim_count")
pivot.plot(kind="bar", stacked=True)
plt.xlabel("Incident Severity")
plt.ylabel("Claim Count")
plt.title("Claim Count by Incident Type and Incident Severity")
plt.legend(title="Incident Type")
plt.show()

For claims where fraud was reported, approximately 94% came from multi-vehicle collisions and single vehicle collisions. In comparison, 21% of non-fraudulent claims came from parked cars and vehicle theft. This suggests that fraudulent claims in this dataset are disproportionately concentrated among collision related incidents, while parked-car and vehicle-theft claims are more common among non-fraudulent claims. Therefore, incident type may be a useful factor when identifying claims that warrant additional fraud screening.

There also appears to be a relationship between incident type and incident severity. Major damage and total loss claims occurred only among multi-vehicle and single vehicle collisions, while other incident types were associated with lower levels of damage. This is important when interpreting the relationship between fraud and incident severity, because the higher proportion of major damage claims among fraudulent claims may be partly related to the types of incidents in which those claims occur. Overall, both incident type and incident severity show differences between fraudulent and non-fraudulent claims in this dataset, although these relationships do not by themselves establish that a claim is fraudulent.

In [ ]:
query = """
SELECT fraud_reported,
collision_type,
COUNT(*) AS claim_count,
COUNT(*)*100.0/SUM(COUNT(*)) OVER (PARTITION BY fraud_reported) AS fraud_rate
FROM claims
GROUP BY collision_type, fraud_reported
ORDER BY collision_type;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

#stacked bar chart
pivot = temp_df.pivot(index="fraud_reported", columns="collision_type", values="claim_count")
pivot.plot(kind="bar", stacked=True)
plt.xlabel("Fraud Reported")
plt.ylabel("Claim Count")
plt.title("Claim Count by Fraud Reported and Collision Type")
plt.legend(title="Collision Type")
plt.show()

Fraudulent claims are much less likely to be "caused" by unknown collisions as compared to non-fradulent claims; 6% of fraudulent claims to 21% of non-fraudulent claims. Additionally, rear collisions were slightly more likely for fradulent claims than for non fradulent. While the other two collision types were nearly identical. This shows that collision type could be a factor that should be considered when identifying claims that warrant fraud screening.


Overall, when considering incident severity, incident type, and collision type together, a clear pattern emerges; fraudulent claims are more likely to involve multi-vehicle and single vehicle collisions with major damage, and less likely to involve unknown collision types. This is consistent with earlier findings that these variables are tightly linked rather than independent: major damage claims consist exclusively of multi-vehicle and single vehicle collisions, and among collision-type incidents, trivial damage occurs exclusively under unknown collisions, while unknown collisions themselves consist exclusively of minor and trivial damage. Separately, trivial damage claims overall are also associated with non-collision incident types like vehicle theft incidents and parked car incidents. Because of this overlap between incident severity, incident type, and collision type, it's difficult to isolate which variable is truly driving the fraud pattern.

3c. Claim Characteristics Associated With Fraud

In [ ]:
query = """
SELECT fraud_reported,
number_of_vehicles_involved,
COUNT(*) AS claim_count,
COUNT(*)*100.0/SUM(COUNT(*)) OVER (PARTITION BY number_of_vehicles_involved) AS fraud_rate
FROM claims
GROUP BY number_of_vehicles_involved, fraud_reported
ORDER BY number_of_vehicles_involved;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

#chi-squared test
contingency = temp_df.pivot(index="number_of_vehicles_involved", columns="fraud_reported", values="claim_count")
chi2, p, dof, expected = chi2_contingency(contingency)
print(f"Chi-square statistic: {chi2:.2f}")
print(f"p-value: {p:.4f}")
print("Expected counts:\n", expected)


Overall, there appears to be no relationship between number of vehicles involved and reported fraud. The fraud rate for four vehicles is slightly higher at 35% vs 30%, 22.9%, and 26.3% for the other number of vehicles. However, this is not a large jump, and given the small sample size of incidents with two and four vehicles; about a 10-15x smaller sample size than three and one vehicle accidents respectively, this increase in fraud could just be due to the small sample size. This was further proven by a Chi-Squared test of independence found no statistically significant relationship between number of vehicles involved and fraud status (χ² = 3.88, p = 0.27), suggesting that, unlike collision type, the number of vehicles involved does not meaningfully predict whether a claim is fraudulent.

In [ ]:
query = """
SELECT fraud_reported,
bodily_injuries,
COUNT(*) AS claim_count,
COUNT(*)*100.0/SUM(COUNT(*)) OVER (PARTITION BY bodily_injuries) AS fraud_rate
FROM claims
GROUP BY bodily_injuries, fraud_reported
ORDER BY bodily_injuries;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

As with number of vehicles involved, no meaningful relationship was found between bodily injuries and reported fraud based on the similarity of fraud rates across groups.

In [ ]:
query = """
SELECT fraud_reported,
property_damage,
COUNT(*) AS claim_count,
COUNT(*)*100.0/SUM(COUNT(*)) OVER (PARTITION BY property_damage) AS fraud_rate
FROM claims
GROUP BY property_damage, fraud_reported
ORDER BY property_damage;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

#bar chart
pivot = temp_df.pivot(index="property_damage", columns="fraud_reported", values="claim_count")
pivot.plot(kind="bar", stacked=True)
plt.xlabel("Property Damage")
plt.ylabel("Claim Count")
plt.title("Claim Count by Property Damage and Fraud")
plt.legend(title="Reported Fraud")
plt.show()

#chi-squared test
contingency = temp_df.pivot(index="property_damage", columns="fraud_reported", values="claim_count")
chi2, p, dof, expected = chi2_contingency(contingency)
print(f"Chi-square statistic: {chi2:.2f}")
print(f"p-value: {p:.4f}")
print("Expected counts:\n", expected)

A chi-square test found a statistically significant relationship between property damage status and fraud (χ² = 8.03, p = 0.018). Fraud rates were lowest for claims with no property damage (19.53%) and highest for claims with unknown property damage status (28.61%), suggesting ambiguity surrounding property damage may be associated with elevated fraud risk.

3d. Policy Characteristics Associated With Fraud?

In [ ]:
query = """
SELECT fraud_reported,
policy_deductable,
COUNT(*) AS claim_count,
COUNT(*)*100.0/SUM(COUNT(*)) OVER (PARTITION BY policy_deductable) AS fraud_rate
FROM claims
GROUP BY policy_deductable, fraud_reported
ORDER BY policy_deductable;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

As with number of vehicles involved, no meaningful relationship was found between policy deductable and reported fraud based on the similarity of fraud rates across groups.

In [ ]:

query = """
SELECT policy_annual_premium
FROM claims;
"""
temp_df = pd.read_sql_query(query, conn)

#distribution of policy annual premium graph
plt.hist(temp_df["policy_annual_premium"], bins="sqrt")
plt.xlabel("Policy Annual Premium")
plt.ylabel("Number of Claims")
plt.title("Distribution of Policy Annual Premium")
plt.show()
temp_df["premium_quartile"] = pd.qcut(temp_df["policy_annual_premium"], q=4, labels=["Low", "Medium-Low", "Medium-High", "High"])

In [ ]:
query = """
WITH premium_groups AS (
    SELECT
        fraud_reported,
        NTILE(4) OVER (ORDER BY policy_annual_premium) AS quartile
    FROM claims
)

SELECT
    quartile,
    fraud_reported,
    COUNT(*) AS claim_count,
    COUNT(*)*100.0/SUM(COUNT(*)) OVER (PARTITION BY quartile) AS fraud_rate
FROM premium_groups
GROUP BY quartile, fraud_reported
ORDER BY quartile;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

#chi-squared test
table = temp_df.pivot(
    index="quartile",
    columns="fraud_reported",
    values="claim_count"
)

chi2, p, dof, expected = chi2_contingency(table)

print("Chi-square statistic:", chi2)
print("p-value:", p)
print("Degrees of freedom:", dof)
print("Expected counts:")
print(expected)


Before comparing fraud and annual policy premiums, I first looked at the distribution of the premiums, which ended up being roughly bell shaped. Then I split the premiums into four quartiles based on size to compare the average claim amounts of them Average claim amounts were roughly similar amongst each quartile of the annual policy premium, however the fraud rate differed. Medium-Low policies had the lowest fraud rate at 19.2%, which was 2/3 the size of the largest fraud rate of 30.4% for medium-high policies. Additionally the chi-squared test of premiums vs fraud had χ² = 9.74, p = 0.021, which means that there is a statistically significant relationship between annual policy premiums and fraud. This implies that there shoulkd perhaps be more scrutiny towards medium-high annual premiums claiming fraud.

Findings of Part 3:

Overall, fraudulent claims have higher average claim amounts, but they account for only about 1/4 of all claims in the dataset. Fraud appears to be associated with incident severity, incident type, and collision type, although it is difficult to separate the relationships between these variables because they also appear to be related to one another. Most of the claim characteristics examined did not show a meaningful relationship with reported fraud, with property damage being an exception. Policy deductible also did not show a meaningful relationship with fraudulent claims, while policy annual premium did show a statistically significant association with fraud. Overall, these findings suggest that incident characteristics, property damage, and annual premium may be useful factors to consider when investigating potentially fraudulent claims, although further analysis would be needed to determine how these factors interact with one another.

Part 4: Geography

Research question: How do claims vary by geography?

4a. Incident State Loss Indicators and Spread

In [ ]:
query = """
SELECT incident_state, 
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY incident_state
ORDER BY incident_state DESC;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

In [ ]:
query = """
SELECT 
    incident_state,
    total_claim_amount
FROM claims;
"""
temp_df = pd.read_sql_query(query, conn)

#one-way ANOVA test
groups = [group["total_claim_amount"].values for name, group in temp_df.groupby("incident_state")]
f_stat, p_value = f_oneway(*groups)
print(f"F-statistic: {f_stat:.2f}")
print(f"p-value: {p_value:.4f}")

#bar chart 
std_by_state = temp_df.groupby("incident_state")["total_claim_amount"].std().reset_index()
std_by_state.columns = ["incident_state", "std_dev"]
std_by_state = std_by_state.sort_values("std_dev", ascending=False)
fig, ax = plt.subplots(figsize=(9, 6))
ax.bar(std_by_state["incident_state"], std_by_state["std_dev"], width=0.6)
ax.set_xlabel("Incident State", fontsize=12)
ax.set_ylabel("Standard Deviation of Claim Amount ($)", fontsize=12)
ax.set_title("Claim Amount Variability by State", fontsize=14, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

Since the average claim amounts weren't obviously very close, nor were they obviously very far, and since sample sizes varied widely (263 vs 23), I decided to run a one-way ANOVA test. The test found no statistically significant difference in average claim amount across states at the conventional threshold (F = 2.01, p = 0.06), though the result approached significance, suggesting a possible weak relationship that this sample size may not be large enough to confirm conclusively. This means that likely most of the variation in total losses is due to variation in the number of claims, rather than claim amounts. Variation also differed state to state, though not by a large amount. However, given the small population of Ohio and Pennsylvania this could be skewed by sample size.

4b. Incident State vs Other Variables Related to Loss

In [ ]:
query = """
SELECT incident_state,
incident_severity,
COUNT(*) AS claim_count,
COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY incident_state) AS percentage_of_state
FROM claims
GROUP BY incident_state, incident_severity
ORDER BY incident_state;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

#stacked bar chart
temp_df = pd.read_sql_query(query, conn)
pivot = temp_df.pivot(index="incident_state", columns="incident_severity", values="claim_count")
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_pct.plot(kind="bar", stacked=True, figsize=(10, 6))
plt.xlabel("Incident State")
plt.ylabel("Percentage of Claims (%)")
plt.title("Incident Severity Distribution by State")
plt.legend(title="Incident Severity")
plt.tight_layout()
plt.show()

Incident severity varies by state, with Ohio standing out in particular. Ohio's percentage of trivial damage claims is less than half that of other states, while its percentage of major damage claims is roughly 1.5- 2X higher. Ohio also has the lowest percentage of minor damage claims and the lowest total loss among all states.

Interestingly, despite having a higher proportion of major damage claims and a lower proportion of trivial claims, a mix that would typically suggest higher average costs, Ohio actually has one of the lowest average claim amounts among all states. This suggests either that Ohio's small sample size (23 claims) is producing unreliable averages, or that severity category alone doesn't consistently predict claim cost across different states.

In [ ]:
query = """
SELECT incident_state,
incident_type,
COUNT(*) AS claim_count,
COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY incident_state) AS percentage_of_state
FROM claims
GROUP BY incident_state, incident_type
ORDER BY incident_state;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

The distribution of incident type also varies by state. New York has the highest percentage of collision-type incidents overall and the lowest percentage of vehicle theft, consistent with collision-type incidents generally having higher claim amounts than non-collision incidents like theft or parked car. Pennsylvania has a notably higher share of parked car claims than every other state, ranging from about 1.4 to 3.25 times higher depending on the state being compared, and along with Ohio, has one of the highest shares of multi-vehicle collisions (~50% and ~52%, respectively). Ohio and Pennsylvania also show a notably smaller share of single-vehicle collisions than other states; roughly 67% to 77% smaller, depending on the comparison state.  Which, since single vehicle collisions tend to have slightly higher average claim amounts, may partially explain their lower average claim amounts. However, given that the ANOVA test comparing claim amount across states returned only a borderline result (p = 0.06), this relationship should be treated as suggestive rather than conclusive.

In [ ]:
query = """
SELECT incident_state,
collision_type,
COUNT(*) AS claim_count,
COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY incident_state) AS percentage_of_state
FROM claims
GROUP BY incident_state, collision_type
ORDER BY incident_state;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

#stacked bar chart
pivot = temp_df.pivot(index="incident_state", columns="collision_type", values="claim_count")
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_pct.plot(kind="bar", stacked=True, figsize=(10, 6))
plt.xlabel("Incident State")
plt.ylabel("Percentage of Claims (%)")
plt.title("Collision Type Distribution by State")
plt.legend(title="Collision Type")
plt.tight_layout()
plt.show()

Unknown collisions are highest in North Carolina, Virginia, and West Virginia, and lowest in New York, with the percentage of unknown collisions in New York being about half the percentage in the former 3 states. This is important as unknown collsions have a much lower average claim amount than other collision types. However, variances in the other 3 collision types is not very remarkable given that average claim amounts between them are very similar. 

In [ ]:
query = """
SELECT incident_state,
fraud_reported,
COUNT(*) AS claim_count,
COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY incident_state) AS percentage_of_state
FROM claims
GROUP BY incident_state, fraud_reported
ORDER BY incident_state;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

#stacked bar chart
pivot = temp_df.pivot(index="incident_state", columns="fraud_reported", values="claim_count")
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_pct.plot(kind="bar", stacked=True, figsize=(10, 6))
plt.xlabel("Incident State")
plt.ylabel("Percentage of Claims (%)")
plt.title("Fraud Distribution by State")
plt.legend(title="Fraud")
plt.tight_layout()
plt.show()

#chi-squared test
contingency = temp_df.pivot(index="incident_state", columns="fraud_reported", values="claim_count")
chi2, p, dof, expected = chi2_contingency(contingency)

print(f"Chi-square statistic: {chi2:.2f}")
print(f"p-value: {p:.4f}")
print("Expected counts:\n", expected)

The fraud percentage is much higher in Ohio, ranging from 1.58x higher to 2.52x higher. This ends up making about 40% of claims in Ohio fraduelent. However, given Ohio's small sample size I decided to double check this result with a Chi-squared test(χ² = 16.13, p = 0.0131). The small p value means there is a statistically significant relationship between fraudulent claims and state, which implies that it may be helpful to double check claims coming from states with a higher percentage of fraudulent claims as part of fraud prevention.

4c. Policy State

In [ ]:
query = """
SELECT policy_state,
incident_state,
COUNT(*) AS claim_count,
AVG(total_claim_amount) AS average_claim_count,
SUM(total_claim_amount) AS total_loss
FROM claims
GROUP BY policy_state, incident_state
ORDER BY policy_state;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

#stacked bar chart
temp_df = pd.read_sql_query(query, conn)
pivot = temp_df.pivot(index="incident_state", columns="policy_state", values="claim_count")
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_pct.plot(kind="bar", stacked=True, figsize=(10, 6))
plt.xlabel("Incident State")
plt.ylabel("Percentage of Claims (%)")
plt.title("Make up of Policy State by Incident State")
plt.legend(title="Policy State")
plt.tight_layout()
plt.show()

In [ ]:
query = """
SELECT policy_state, 
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY policy_state
ORDER BY policy_state DESC;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

Average claim amount appears to be very similar across states, as does claim count, and total losses. However the makeup of incident states by policy states does appear to differ in Virginia, with Virginia having much more policies from Indiana than other states. This is not a likley explanation of any potential differences in average claims however, since policy state does not appear to have much of a meaningful relationship with loss metrics. 

4d. Incident City

In [ ]:
query = """
SELECT incident_city, 
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY incident_city
ORDER BY incident_city DESC;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

In [ ]:
query = """
SELECT incident_state, incident_city, COUNT(*) AS claim_count
FROM claims
GROUP BY incident_state, incident_city
ORDER BY incident_state, incident_city;
"""
temp_df = pd.read_sql_query(query,conn)
show_formatted(temp_df)

The same seven city names appear across multiple states, so city does not identify a unique location, and city-level results combine claims from different states. Average claim amounts were very similar across the seven cities (about 48,500 to 54,500), so city does not appear to be a strong driver of claim amount. Because city and state appear to have been assigned independently, state is a better indicator of geography, and the small number of claims in some states (such as OH and PA) limits conclusions about them. 

4e. Incident Location and Insured Zip

These fields contained individual street addresses and individual zip codes respectvely, both of which I found to be unique to nearly every claim, so location was analyzed at the state level instead.

Pt 4 conclusion: 

Incident state is the only geographic variable that showed any sort of difference in claims, though in this case the result was borderline. Ohio specifically exhibited a statistically significant higher fraud rate (40% vs. 18–31%, p = 0.0131) and a contradictory profile, higher major damage severity paired with low average claim amounts and a small sample size (n = 23). However, incident location, zip, and incident city showed no relationship and in this case incident city was actually assigned independently of state, thus not being a good variable to compare. This may suggest that there should be larger sample sizes gathered of states such as Ohio to accurately understand the differences in claims.

Part 5: Policy Holder Characteristics

Research Question: Are certain policyholder characteristics associated with differences in claim amounts or claim frequency?

5a. Age

In [ ]:
query = """
SELECT age, 
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY age
ORDER BY age DESC;
"""
temp_df = pd.read_sql_query(query, conn)

#scatter plot
plt.scatter(temp_df["age"], temp_df["average_claim_amount"])
plt.xlabel("Age")
plt.ylabel("Average Claim Amount")
plt.title("Average Claim Amount by Age")
plt.show()

In [ ]:
query = """
SELECT age,
total_claim_amount
FROM claims;
"""
temp_df = pd.read_sql_query(query,conn)
#correlation coeff
corr, p_value = pearsonr(temp_df["age"], temp_df["total_claim_amount"])
print(f"Correlation coefficient: {corr:.3f}")
print(f"p-value: {p_value:.4f}")

#histogram 
plt.hist(temp_df["age"], bins="sqrt", edgecolor="black")
plt.title("Distribution of Age")
plt.xlabel("Age")
plt.ylabel("Number of Claims")
plt.show()

Age is approximately unimodal, centered in the late 30s to early 40s, with a right-skewed tail toward older ages. The correlation between age and claim amount is 0.070 (p = 0.027), a statistically significant but very weak positive relationship. The scatter plot of age against average claim amount shows a slight upward trend, with outliers around 20 and 60. Age is therefore unlikely to be a useful predictor of claim severity on its own.

5b. Gender

In [ ]:
query = """
SELECT insured_sex, 
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY insured_sex
ORDER BY insured_sex;
"""
claims_df = pd.read_sql_query(query, conn)
show_formatted(claims_df)


The average claim amount is very similar regardless of gender. However females have slightly more claims than males (1.15x more). However, without total policies it is hard to tell if they have a higher frequency of claims than males.

5c. Education Level

In [ ]:
query = """
SELECT insured_education_level,
total_claim_amount,
COUNT(*) AS claim_count,
AVG(total_claim_amount) AS average_claim_amount,
SUM(total_claim_amount) AS total_loss
FROM claims
GROUP BY insured_education_level
ORDER by insured_education_level;
"""
temp_df = pd.read_sql_query(query,conn)
show_formatted(temp_df)



In [ ]:
query = """
SELECT insured_education_level,
total_claim_amount
FROM claims;
"""
temp_df = pd.read_sql_query(query,conn)

#one way ANOVA test
groups = [group["total_claim_amount"].values for name, group in temp_df.groupby("insured_education_level")]
f_stat, p_value = f_oneway(*groups)
print(f"F-statistic: {f_stat:.2f}")
print(f"p-value: {p_value:.4f}")

There appeared to be some possible variations in average claim amounts between each education level. So I decided to run an ANOVA test comparing education level against claim amounts(F= 1.14,
p= 0.3346). This means there was no statistically significant difference between average claim amounts for each education level. Similarly, claim counts appeared to also be relatively uniformly dictributed. 

5d. Occupation

In [ ]:
query = """
SELECT insured_occupation,
total_claim_amount,
COUNT(*) AS claim_count,
AVG(total_claim_amount) AS average_claim_amount,
SUM(total_claim_amount) AS total_loss
FROM claims
GROUP BY insured_occupation
ORDER by insured_occupation;
"""
temp_df = pd.read_sql_query(query,conn)
show_formatted(temp_df)

In [ ]:
query = """
SELECT insured_occupation, total_claim_amount
FROM claims;
"""
claims_df = pd.read_sql_query(query, conn)

groups = [group["total_claim_amount"].values for name, group in claims_df.groupby("insured_occupation")]

f_stat, p_value = f_oneway(*groups)

print(f"F-statistic: {f_stat:.2f}")
print(f"p-value: {p_value:.4f}")

Similar to education level, there appeared to be some possible variations in average claim amounts between each occupation. So I decided to run an ANOVA test comparing education level against claim amounts(F= 1.31,p= 0.2009). This means there was no statistically significant difference between average claim amounts for each occupation level. Similarly, claim counts appeared to also be relatively uniformly dictributed. 

5e. Relationship Type

In [ ]:
query = """
SELECT insured_relationship, 
COUNT(*) AS claim_count,
AVG(total_claim_amount) AS average_claim_amount,
SUM(total_claim_amount) AS total_loss
FROM claims
GROUP BY insured_relationship
ORDER BY insured_relationship;
"""
temp_df = pd.read_sql_query(query,conn)
show_formatted(temp_df)


The differences in average claim amounts and claim counts by relationship type are so small, they don't seem to warrant testing via ANOVA. It can be concluded that there likely is no relationship between loss indicators and relationship type.

5f. Months as a Customer

In [ ]:
query = """
SELECT months_as_customer, 
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY months_as_customer
ORDER BY months_as_customer;
"""
temp_df = pd.read_sql_query(query, conn)

#scatter plot
plt.scatter(temp_df["months_as_customer"], temp_df["average_claim_amount"])
plt.xlabel("Months as Customer")
plt.ylabel("Average Claim Amount")
plt.title("Average Claim Amount by Months as Customer")
plt.show()

In [ ]:
query = """
SELECT months_as_customer,
total_claim_amount
FROM claims;
"""
temp_df = pd.read_sql_query(query, conn)

#correlation test
corr, p_value = pearsonr(temp_df["months_as_customer"], temp_df["total_claim_amount"])
print(f"Correlation coefficient: {corr:.3f}")
print(f"p-value: {p_value:.4f}")

#histogram injury claims
plt.hist(temp_df["months_as_customer"], bins="sqrt", edgecolor="black")
plt.title("Distribution of Months as a Customer")
plt.xlabel("Months as a Customer")
plt.ylabel("Number of Claims")
plt.show()

Months as customer showed a very weak but borderline statistically significant positive correlation with average claim amount (r = 0.062, p = 0.049), suggesting a slight tendency for longer-tenured customers to file somewhat larger claims: though this relationship explains only a small portion of the overall variation in claim amounts. There is a sizeable chunk of claims with low average claim amounts regardless of months as customer. Months as a customer is a single broad distribution around 100-300 months with a slight drop around 200 months and with a rightward tail after 300 customers.

5g. Capital Gains and Capital Loss

In [ ]:
query = """
SELECT "capital-gains", 
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY "capital-gains"
ORDER BY "capital-gains";
"""
temp_df = pd.read_sql_query(query, conn)

#scatter plot
plt.scatter(temp_df["capital-gains"], temp_df["average_claim_amount"])
plt.xlabel("Capital Gains")
plt.ylabel("Average Claim Amount($)")
plt.title("Average Claim Amount by Capital Gains")
plt.show()

In [ ]:
query = """
SELECT "capital-gains", 
total_claim_amount
FROM claims;
"""
temp_df = pd.read_sql_query(query, conn)

#correlation
corr = temp_df["capital-gains"].corr(temp_df["total_claim_amount"])
print(f"Correlation coefficient: {corr:.3f}")

The scatter plot of capital gains against average claim amount shows no visible trend. This is consistent with the correlation between capital gains and individual claim amounts (r = 0.016), which is essentially zero. Together, these suggest little to no linear relationship between capital gains and claim amount.

Part 5 conclusions: 

While months as a customer and age showed very weak but statistically significant positive relationships with average claim amount, no other policyholder characteristics tested showed a meaningful relationship with loss indicators. This suggests that, within this dataset, policyholder characteristics are not strongly associated with claim severity; though this alone doesn't necessarily mean they should be excluded from pricing decisions, since factors like fraud risk, retention, or regulatory considerations may still justify their use elsewhere in the pricing model.

Part 6: Policy Characteristics

Research Question: Do characteristics of policies affect loss indicators?

6a. Deductible

In [ ]:
query = """
SELECT policy_deductable, 
AVG(total_claim_amount) AS average_severity, 
COUNT(*) AS claim_count,
COUNT(*)*100/(SELECT COUNT(*) FROM claims) AS claim_percentage,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY policy_deductable
ORDER BY policy_deductable DESC;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

In [ ]:
query = """
SELECT policy_deductable, 
incident_type,
COUNT(*) AS claim_count,
COUNT(*)*100.0/ SUM(COUNT(*)) OVER (PARTITION BY policy_deductable) AS percentage_within_policy_deductable
FROM claims
GROUP BY policy_deductable, incident_type
ORDER BY policy_deductable, incident_type DESC;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

In [ ]:
query = """
SELECT policy_deductable,
total_claim_amount
FROM claims;
"""
temp_df = pd.read_sql_query(query, conn)

#correlation test
correlation = temp_df["policy_deductable"].corr(temp_df[ "total_claim_amount"])
print(f"Correlation between policy deductable and total claim amount:{correlation:.3f}")

Claim percentages were slightly lower for policies with a deductible of 2000, while they were relatively similar for deductibles of 500 and 1000. Average claim severity was slightly higher for the 2000 deductible and lowest for the 1000 deductible, but average severity was generally similar across all three deductible levels. The distribution of incident severity and the types of incidents were also relatively similar across deductible levels, although the 2000 deductible had slightly more multi-vehicle collisions and slightly fewer parked-car incidents. Additionally the correlation of about 0.023 between policy deductable and total loss further shows there is little to no relationship between deductable and total loss. Overall, these results suggest that policy deductible was not strongly associated with claim severity in this dataset, nor was it strongly associated with incident type nor incident severity.

6b. Policy Annual Premium

In [ ]:
query = """
SELECT policy_annual_premium, 
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY policy_annual_premium
ORDER BY policy_annual_premium;
"""
temp_df = pd.read_sql_query(query, conn)

#scatter plot
plt.scatter(temp_df["policy_annual_premium"], temp_df["average_claim_amount"])
plt.xlabel("Annual Premium")
plt.ylabel("Average Claim Amount")
plt.title("Average Claim Amount by Annual Premium")
plt.show()

In [ ]:
query = """
SELECT incident_type, policy_annual_premium, total_claim_amount
FROM claims;
"""
temp_df = pd.read_sql_query(query, conn)

#correlation test
for incident_type, group in temp_df.groupby("incident_type"):
    corr, p_value = pearsonr(group["policy_annual_premium"], group["total_claim_amount"])
    print(f"{incident_type}: r = {corr:.3f}, p = {p_value:.4f}, n = {len(group)}")

Policy annual premium showed no meaningful relationship with claim amount, even after splitting by incident type (all |r| < 0.06, all p > 0.05). This suggests that within this dataset, how much a policyholder pays annually does not correlate with how large their eventual claims are, regardless of the type of incident involved.

6c. Policy Bind Date

In [ ]:
query = """
SELECT strftime('%Y', policy_bind_date) AS bind_year,
COUNT(*) AS claim_count,
AVG(total_claim_amount) AS average_claim_amount,
SUM(total_claim_amount) AS total_loss
FROM claims
GROUP BY bind_year
ORDER BY bind_year;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

#scatterplot
plt.scatter(temp_df["bind_year"], temp_df["average_claim_amount"])
plt.xlabel("Policy Bind Year")
plt.ylabel("Average Claim Amount")
plt.title("Average Claim Amount by Policy Bind Year")
plt.xticks(rotation=45, ha="right")
plt.show()

There appears to not be a linear relationship between average claim amount and policy year. With average claim amount varying between 47,000 and 60,000. There is a notable outlier at 2015, however this is likely due to the small sample size of 3 claims which is more than 10x smaller than claims under any other year.

6d. Umbrella Limit

In [ ]:
query = """
SELECT umbrella_limit, 
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY umbrella_limit
ORDER BY umbrella_limit;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

#scatter plot
plt.scatter(temp_df["umbrella_limit"], temp_df["average_claim_amount"])
plt.xlabel("Umbrella Limit")
plt.ylabel("Average Claim Amount")
plt.title("Average Claim Amount by Umbrella Limit")
plt.show()

In [ ]:
query = """
SELECT 
    CASE WHEN umbrella_limit > 0 THEN 'Has Umbrella' ELSE 'No Umbrella' END AS umbrella_group,
    total_claim_amount
FROM claims
WHERE umbrella_limit >= 0;
"""
temp_df = pd.read_sql_query(query, conn)

#t-test
has = temp_df[temp_df["umbrella_group"] == "Has Umbrella"]["total_claim_amount"]
no = temp_df[temp_df["umbrella_group"] == "No Umbrella"]["total_claim_amount"]
t_stat, p_value = ttest_ind(has, no) 
print(f"T-statistic: {t_stat:.2f}")
print(f"p-value: {p_value:.4f}")

From the scatter plot there would appear to be a very large outlier at a negative umberella limit, however this is not possible since umbrealla limits are extra coverage, so they cannot be negative, thus this value must be due to an error in the data. I decided to run a t-test to see if there were any statistically significant differences when excluding the negative umbrella amount and I got a T-statistic: -1.10, p-value: 0.2708, which implies there is not any statistically significant relationship between umbrella limits and average claim amounts.

6e. Policy Combined Single Limit

In [ ]:
query = """
SELECT policy_csl, 
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY policy_csl
ORDER BY policy_csl;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

#bar chart
plt.bar(temp_df["policy_csl"], temp_df["average_claim_amount"])
plt.xlabel("Policy Combined Single Limit")
plt.ylabel("Average Claim Amount ($)")
plt.title("Average Claim Amount by Policy Combined Single Limit")
plt.show()


In [ ]:
query = """
SELECT policy_csl,
total_claim_amount
FROM claims;
"""
temp_df = pd.read_sql_query(query, conn)

#one way ANOVA test
groups = [group["total_claim_amount"].values for name, group in temp_df.groupby("policy_csl")]
f_stat, p_value = f_oneway(*groups)
print(f"F-statistic: {f_stat:.2f}")
print(f"p-value: {p_value:.4f}")


There appeared to be some variation between average claim amount and policy combined single limit. With the lowest, policy combined single limits (100/300) being higher by about 1.08x than both of the other combined single limits. I ran an ANOVA test (F-statistic: 2.45647908677055
p-value: 0.08625506932700214) and determined there is not a statistically significant difference in average claim amount for combined single limits.

Part 6 conclusion: 

None of the five policy characteristics (deductible, annual premium, policy year, umbrella limit, policy combined single limit) showed a meaningful relationship with claim size or incident characteristics. Deductible differences were small (correlation with total loss ≈ 0.023), premium was unrelated to claim amount (all |r| < 0.06), policy year showed no linear trend (the 2015 outlier reflects only 3 claims), umbrella limit had no significant effect once a single invalid negative entry was removed (p = 0.2708), and single policy combined limit did not share a significant relationship with claim amounts. Overall, these variables are not strong predictors of claim size in this dataset. Future work could test them in combination with multivariate models and add factors like driver demographics or incident circumstances.

Part 7: Vehicle Characteristics

Research Question: Are there specific aspects of a vehicle that might lead to different claim amounts?

7a. Car Brand

In [ ]:
query = """
SELECT auto_make, 
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY auto_make
ORDER BY auto_make;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

In [ ]:
query = """
SELECT auto_make,
total_claim_amount
FROM claims;
"""
temp_df = pd.read_sql_query(query, conn)

#one way ANOVA test
groups = [group["total_claim_amount"].values for name, group in temp_df.groupby("auto_make")]
f_stat, p_value = f_oneway(*groups)
print(f"F-statistic: {f_stat:.2f}")
print(f"p-value: {p_value:.4f}")

Average claim amounts between car brand are very close, with the exception of Toyota cars which had a lower average claim amount relative to interbrand differences. Claim count is also similar, with slightly less claims from Honda, which may be the cause of its low total losses relative to other brands. Running a one way ANOVA test yeilds a F-statistic: 0.68 and a p-value: 0.7817. This means there is no significant relationship between car brand and loss indicators. This implies that the variations in total loss are mainly due to differences in claim counts, not any underlying difference in average claim amount.

7b. Car Model

In [ ]:
query = """
SELECT auto_model, 
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY auto_model
ORDER BY auto_model;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

#bar chart
plt.bar(temp_df["auto_model"], temp_df["average_claim_amount"])
plt.xlabel("Auto Model")
plt.ylabel("Average Claim Amount")
plt.title("Average Claim Amount by Auto Model")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
query = """
SELECT auto_model,
total_claim_amount
FROM claims;
"""
temp_df = pd.read_sql_query(query, conn)

#one way ANOVA test
groups = [group["total_claim_amount"].values for name, group in temp_df.groupby("auto_model")]
f_stat, p_value = f_oneway(*groups)
print(f"F-statistic: {f_stat:.2f}")
print(f"p-value: {p_value:.4f}")

The chart and graph themselves show some differences in average claim amount across various auto models. However, after running a one way ANOVA test (F= 0.97, p = 0.5175), it can be concluded that there is no statistically significant relationship between car model and claim amount. 

7c. Car Age

In [ ]:
query = """
SELECT auto_year, 
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY auto_year
ORDER BY auto_year;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

In [ ]:
query = """
SELECT auto_year,
total_claim_amount
FROM claims;
"""
temp_df = pd.read_sql_query(query, conn)

#one way ANOVA test
groups = [group["total_claim_amount"].values for name, group in temp_df.groupby("auto_year")]
f_stat, p_value = f_oneway(*groups)
print(f"F-statistic: {f_stat:.2f}")
print(f"p-value: {p_value:.4f}")

The differences in average claim amounts across car age strongly resembles that seen in car model and car brand. Running an ANOVA test (F-statistic: 0.96, p-value: 0.51) confirms that there is no statistically significant relationship between car age and claim amount.

Part 7 conclusion: 

Overall, the vehicle characteristics examined (auto model, brand, and age) showed no statistically detectable relationship with claim severity. One-way ANOVA tests found no significant differences in average claim amount across any of the three (all p > 0.5), and average claim amounts were similar across groups. Differences in total loss were driven mainly by how many claims each group had rather than by the size of individual claims. Based on this data, vehicle characteristics appear to be a lower priority than other factors when assessing loss.

Part 8: Time characteristics

8a. Claims by Hours of the Day

In [ ]:
query = """
SELECT incident_hour_of_the_day, 
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY incident_hour_of_the_day
ORDER BY incident_hour_of_the_day;
"""
temp_df = pd.read_sql_query(query, conn)
show_formatted(temp_df)

#scatter plot
plt.scatter(temp_df["incident_hour_of_the_day"], temp_df["average_claim_amount"])
plt.xlabel("Incident Hour of the Day")
plt.ylabel("Average Claim Amount")
plt.title("Average Claim Amount by Hour of the Day")
plt.show()

In [ ]:
query = """
SELECT incident_hour_of_the_day, 
total_claim_amount
FROM claims;
"""
temp_df = pd.read_sql_query(query, conn)
#correlation 
corr = temp_df["incident_hour_of_the_day"].corr(temp_df["total_claim_amount"])
print(f"Correlation coefficient: {corr:.3f}")


In [ ]:
query = """
SELECT incident_hour_of_the_day, 
fraud_reported,
AVG(total_claim_amount) AS average_claim_amount, 
COUNT(*) AS claim_count,
SUM(total_claim_amount) AS total_losses
FROM claims
GROUP BY incident_hour_of_the_day, fraud_reported
ORDER BY incident_hour_of_the_day;
"""
temp_df = pd.read_sql_query(query, conn)

#stacked bar chart
pivot = temp_df.pivot(index="incident_hour_of_the_day", columns="fraud_reported", values="claim_count")
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_pct.plot(kind="bar", stacked=True, figsize=(10, 6))
plt.xlabel("Hour of the Day by Incident")
plt.ylabel("Percentage of Claims (%)")
plt.title("Fraud Distribution by Hour of the Day")
plt.legend(title="Fraud")
plt.tight_layout()
plt.show()

Average claim amount differs in clusters by time of day. Average claim amount is around 55,000 or higher for incidents between 10am to 2am. From 3am to 9am average claim amount falls to around 45,000 or lower. This is supported by the correlation coefficient of 0.218, indicating a weak positive relationship between time of day and average claim amount. I also compared fraud to see if it could possibly correlate with these higher average claim amount times, but fraud appeared to be randomly distributed amongst hour with no clear pattern. 

8b. Incident Month and Incident Year

In [ ]:
query = """
SELECT strftime('%m', incident_date) AS incident_month,
        COUNT(*) AS claim_count,
       AVG(total_claim_amount) AS average_claim_amount,
       SUM(total_claim_amount) AS total_loss
FROM claims
GROUP BY incident_month
ORDER BY incident_month;
"""

claims_df = pd.read_sql_query(query, conn)
show_formatted(claims_df)

The data set only contains incidents that occur for 3 months over 1 year, with one month, March, only having 12 claim counts. This does not include enough months or years of claim data to find any meaningful relationship between the month and years and loss indicators.

Part 8 conclusion: 

Overall, average claim amounts show a weak positive correlation with time of day (r = 0.218), with incidents occuring between 2am to 10am having lower average claim amounts, however there was no clear pattern between this and fraud. Additionally, it was insufficient to establish meaningful relationships between months/years and loss indicators as the data set only spanned three months out of one year. This could possibly point to the benefits of factoring in the time of day when making prices models.